# Routes on a multimodal network

A **multimodal network** joins roads, railways, sea lanes and flights. This notebook loads the network `oceania` and answers four questions.

You need scgraph 4.0 or newer, and [folium](https://python-visualization.github.io/folium/) for the map. The first run downloads the network once (about 10 MB). GitHub does not show the map: run the notebook, or open it in [nbviewer](https://nbviewer.org/github/connor-makowski/scgraph/blob/main/examples/multimodal_networks.ipynb).

In [1]:
# Until scgraph 4.0 is on PyPI, install it from the `dev` branch:
# %pip install "scgraph @ git+https://github.com/connor-makowski/scgraph@dev" folium
import folium

from scgraph import GeoGraph

## 1. What does this network hold?

`oceania` holds the main roads and the railways of Australia and New Zealand. It also holds every sea lane, port and airport of the world. Each node has a **node mode**:

In [2]:
oceania = GeoGraph.load_geograph("oceania")
print([node_mode["name"] for node_mode in oceania.node_modes])

['road', 'rail', 'sea', 'port', 'airport', 'rail_terminal']


A port, an airport or a rail terminal is a **hub**: a route can change mode there.

Each edge has a **class**, and each class has an **edge mode**: `road`, `rail`, `sea` or `air`.

In [3]:
for edge_mode in ["road", "rail", "sea", "air"]:
    names = [row["name"] for row in oceania.classes if row["edge_mode"] == edge_mode]
    print(f"{edge_mode:<5}", ", ".join(names))

road  motorway, motorway_link, trunk, trunk_link, primary, primary_link, ferry, shuttle_train, road_connector
rail  rail_highspeed, rail_main, rail_branch, rail_other, rail_connector
sea   open_sea, babelmandeb, bering, corinth, dover, gibraltar, kiel, magellan, malacca, panama, suez, sea_connector
air   flight


## 2. What is the shortest route from Sydney to Perth?

scgraph joins the origin and the destination to their closest road node, with a straight line.

A **leg** is a part of a route on one edge mode. `output_legs=True` returns the legs. The first and the last leg are the two off-graph legs, the straight lines: their mode is `off_graph`.

In [4]:
SYDNEY = {"latitude": -33.8688, "longitude": 151.2093}
PERTH = {"latitude": -31.9523, "longitude": 115.8613}

shortest = oceania.get_shortest_path(origin_node=SYDNEY, destination_node=PERTH, output_legs=True)
print(f"Length: {shortest['length']:,.0f} km")
for leg in shortest["legs"]:
    print(f"{leg['mode']:<10}{leg['length']:>8,.1f} km")

Length: 3,312 km
off_graph      0.5 km
road          17.7 km
air        3,277.2 km
road          17.0 km
off_graph      0.0 km


The route takes a flight. A flight is nearly a straight line, so the shortest route of a long trip often takes one.

## 3. And by road only?

A **metric** gives each edge a **weight**, under a name. A query by a metric returns the route with the smallest total of weights.

`create_metric` computes the weights from **weights per km**: a weight per km is the weight of 1 km. A weight per km of `None` closes an edge mode: no route by this metric takes its edges.

The metric below counts the km of road, and closes the other edge modes.

In [5]:
# "off_graph" is the weight per km of the two off-graph legs: the straight lines at the ends of a route
oceania.create_metric("road_km", weight_per_km={"road": 1, "off_graph": 1, "rail": None, "sea": None, "air": None})

by_road = oceania.get_shortest_path(origin_node=SYDNEY, destination_node=PERTH, metric="road_km", output_legs=True)
print(f"Length: {by_road['length']:,.0f} km")
for leg in by_road["legs"]:
    print(f"{leg['mode']:<10}{leg['length']:>8,.1f} km")

Length: 3,931 km
off_graph      0.5 km
road       3,930.6 km
off_graph      0.0 km


One loaded network answers the two queries. The map draws each leg in the color of its mode: orange for the flight, purple for the roads.

In [6]:
COLORS = {"road": "#882255", "rail": "#009E73", "sea": "#0072B2", "air": "#D55E00", "off_graph": "#555555"}

route_map = folium.Map(location=[-31, 134], zoom_start=4)
for route in [shortest, by_road]:
    for leg in route["legs"]:
        folium.PolyLine(leg["coordinate_path"], color=COLORS[leg["mode"]], tooltip=leg["mode"]).add_to(route_map)
route_map

## 4. Is the shortest route the fastest?

The weights per km of a metric of hours are hours for 1 km: `1 / 80` is one hour for 80 km. `weight_at_hub` adds a weight at a hub: here a flight waits 3 hours at each of its two airports.

In [7]:
# These weights are examples. scgraph ships no speed and no wait: give your own.
oceania.create_metric(
    "hours",
    weight_per_km={"road": 1 / 80, "rail": 1 / 60, "sea": 1 / 30, "air": 1 / 750, "off_graph": 1 / 30},
    weight_at_hub={"airport": {"air": 3}, "port": {"sea": 12}, "rail_terminal": {"rail": 4}},
)

fastest = oceania.get_shortest_path(origin_node=SYDNEY, destination_node=PERTH, metric="hours", output_legs=True)
print(f"{fastest['length']:,.0f} km in {fastest['metric_totals']['hours']:.1f} hours")
for leg in fastest["legs"]:
    print(f"{leg['mode']:<10}{leg['length']:>8,.1f} km{leg['metric_totals']['hours']:>6.1f} h")

3,312 km in 10.8 hours
off_graph      0.5 km   0.0 h
road          17.7 km   0.2 h
air        3,277.2 km  10.4 h
road          17.0 km   0.2 h
off_graph      0.0 km   0.0 h


`metrics` holds the total of each metric along the route. Each leg has its own totals, and the flight leg holds the two waits. Here the fastest route is also the shortest route.

A short trip gives another answer:

In [8]:
CANBERRA = {"latitude": -35.2809, "longitude": 149.1300}

routes = {
    "Shortest": oceania.get_shortest_path(origin_node=SYDNEY, destination_node=CANBERRA, output_legs=True),
    "Fastest": oceania.get_shortest_path(origin_node=SYDNEY, destination_node=CANBERRA, metric="hours", output_legs=True),
}
for name, route in routes.items():
    leg_modes = " + ".join(leg["mode"] for leg in route["legs"][1:-1])
    print(f"{name}: {route['length']:,.0f} km, {route['metric_totals']['hours']:.1f} hours, by {leg_modes}")

Shortest: 265 km, 6.7 hours, by road + air + road
Fastest: 286 km, 3.6 hours, by road


The shortest route still takes a flight. The fastest route stays on the road: the two waits cost more hours than the flight saves.

## What to remember

- `node_modes` lists the node modes, and `classes` lists the classes with their edge modes.
- `output_legs=True` returns the legs of a route.
- Without a metric, a route is the shortest in km. On a long trip, it often takes a flight.
- `create_metric` creates a metric from weights per km. A weight per km of `None` closes an edge mode.
- scgraph ships no speed, no cost and no emission factor: you give the weights.

Next: the notebook *Trade-offs between time, cost and CO2*.